# Follow-up experiments (conf-1, second review)

1. Native `preprocess_input` for VGG16 and ResNet50 (PF and FT, seeds 0-4).
2. Mask-derived silhouette control and whole-image control (DenseNet121-PF, VGG16-PF, seeds 0-2).
3. Determinism check: VGG16-PF seed 0 twice with op determinism enabled.

Run cells 1-5 in order. Results go to `MyDrive/BreastCancerNew/multiseed_v1/runs/`.

In [ ]:
# ===== CELL 1: setup, paths, environment log =====
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, glob, shutil, platform, subprocess, random
import numpy as np, cv2, tensorflow as tf, sklearn
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/BreastCancerNew"
OLD_PROCESSED = f"{DRIVE_BASE}/processed_images"          # existing masked crops (437 benign / 210 malignant)
OUT = f"{DRIVE_BASE}/multiseed_v1"                         # new, never deleted
LOCAL = "/content/ms"
for d in [OUT, f"{OUT}/runs", f"{OUT}/diagnostics", LOCAL]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE, BATCH, EPOCHS, PATIENCE = 224, 32, 50, 20
SPLIT_SEED = 42

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
ENV = {"python": sys.version.split()[0], "tensorflow": tf.__version__, "keras": tf.keras.__version__ if hasattr(tf.keras, "__version__") else "n/a",
       "numpy": np.__version__, "opencv": cv2.__version__, "sklearn": sklearn.__version__,
       "gpu": gpu, "platform": platform.platform(), "time": time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(ENV, open(f"{OUT}/environment_{time.strftime('%Y%m%d_%H%M%S')}.json", "w"), indent=1)
print(json.dumps(ENV, indent=1))
assert tf.config.list_physical_devices('GPU'), "No GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
# ===== CELL 2: data (reuse the masked crops on Drive; same split rule as the paper) =====
# The crops in processed_images were produced by the paper's preprocessing (mask binarized at 127,
# extra-lesion pixels zeroed, 5% padded box, square, 224x224). If they are missing, stop: this
# notebook deliberately reuses them so that all runs see identical images.
counts = {c: len([f for f in os.listdir(f"{OLD_PROCESSED}/{c}") if f.endswith('.png')]) for c in ["benign", "malignant"]}
print("processed images on Drive:", counts)
assert counts == {"benign": 437, "malignant": 210}, "processed_images incomplete"

DATA = f"{LOCAL}/BUSI_split"
def build_split():
    rows = []
    for c in ["benign", "malignant"]:
        imgs = sorted(f for f in os.listdir(f"{OLD_PROCESSED}/{c}") if f.endswith('.png'))
        random.seed(SPLIT_SEED); random.shuffle(imgs)          # identical to the original split_dataset()
        n = len(imgs); t_end, v_end = int(n * 0.70), int(n * 0.85)
        for split, part in (("train", imgs[:t_end]), ("val", imgs[t_end:v_end]), ("test", imgs[v_end:])):
            dst = f"{DATA}/{split}/{c}"; os.makedirs(dst, exist_ok=True)
            for f in part:
                shutil.copy(f"{OLD_PROCESSED}/{c}/{f}", f"{dst}/{f}")
                rows.append((split, c, f))
    return rows
if not os.path.exists(f"{DATA}/test/malignant"):
    rows = build_split()
    with open(f"{OUT}/split.csv", "w") as fh:
        fh.write("split,class,filename\n" + "".join(f"{s},{c},{f}\n" for s, c, f in rows))
for s in ["train", "val", "test"]:
    print(s, {c: len(os.listdir(f"{DATA}/{s}/{c}")) for c in ["benign", "malignant"]})

# shape-only copy: every non-zero pixel -> 255 (lesion silhouette, no texture)
SHAPE = f"{LOCAL}/BUSI_shape"
if not os.path.exists(f"{SHAPE}/test/malignant"):
    for s in ["train", "val", "test"]:
        for c in ["benign", "malignant"]:
            os.makedirs(f"{SHAPE}/{s}/{c}", exist_ok=True)
            for f in os.listdir(f"{DATA}/{s}/{c}"):
                im = cv2.imread(f"{DATA}/{s}/{c}/{f}")
                sil = ((im.max(axis=2) > 0) * 255).astype(np.uint8)
                cv2.imwrite(f"{SHAPE}/{s}/{c}/{f}", cv2.merge([sil, sil, sil]))
print("shape-only images ready")

In [ ]:
# ===== CELL 3: helpers (generators, model, evaluation) =====
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import DenseNet121, Xception, InceptionV3, ResNet50, VGG16
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import roc_auc_score
from scipy.stats import binomtest

BACKBONES = {"DenseNet121": DenseNet121, "Xception": Xception, "InceptionV3": InceptionV3,
             "ResNet50": ResNet50, "VGG16": VGG16}
CFG = {  # identical to the paper (run B/C learning rates)
    "PF": {"freeze": 0.50, "freeze_vgg": 0.60, "lr": 1e-4},
    "FT": {"freeze": 0.40, "freeze_vgg": 0.50, "lr": 1e-5},
}

def gens(root, seed, scale=1/255.):
    aug = ImageDataGenerator(rescale=scale, horizontal_flip=True, vertical_flip=True, zoom_range=0.20,
                             rotation_range=15, width_shift_range=0.10, height_shift_range=0.10, fill_mode='nearest')
    plain = ImageDataGenerator(rescale=scale)
    kw = dict(target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, class_mode='binary', classes=['benign', 'malignant'])
    tr = aug.flow_from_directory(f"{root}/train", shuffle=True, seed=seed, **kw)
    va = plain.flow_from_directory(f"{root}/val", shuffle=False, **kw)
    te = plain.flow_from_directory(f"{root}/test", shuffle=False, **kw)
    return tr, va, te

def build(backbone, strategy):
    c = CFG[strategy]
    base = BACKBONES[backbone](include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    frac = c["freeze_vgg"] if backbone == "VGG16" else c["freeze"]
    k = int(len(base.layers) * frac)
    for i, l in enumerate(base.layers):
        l.trainable = i >= k
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(512, activation='relu')(x); x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x); x = layers.Dropout(0.3)(x)
    m = Model(base.input, layers.Dense(1, activation='sigmoid')(x))
    m.compile(optimizer=tf.keras.optimizers.Adam(c["lr"]), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auroc')])
    return m, frac, c["lr"]

def youden(y, p):
    u = np.unique(p); cuts = np.concatenate(([u[0] - 1e-6], (u[:-1] + u[1:]) / 2, [u[-1] + 1e-6]))
    j = np.array([(p[y == 1] >= t).mean() + (p[y == 0] < t).mean() - 1 for t in cuts])
    return float(np.median(cuts[np.isclose(j, j.max())]))

def counts(y, p, t):
    pr = (p >= t).astype(int)
    return {"tp": int(((pr == 1) & (y == 1)).sum()), "tn": int(((pr == 0) & (y == 0)).sum()),
            "fp": int(((pr == 1) & (y == 0)).sum()), "fn": int(((pr == 0) & (y == 1)).sum())}

def evaluate(model, va, te):
    va.reset(); te.reset()
    pv = model.predict(va, verbose=0).ravel(); pt = model.predict(te, verbose=0).ravel()
    yv, yt = va.classes.astype(int), te.classes.astype(int)
    t = youden(yv, pv)
    return {"val_auroc": float(roc_auc_score(yv, pv)), "test_auroc": float(roc_auc_score(yt, pt)),
            "val_at_0.5": counts(yv, pv, 0.5), "test_at_0.5": counts(yt, pt, 0.5),
            "thr_val": t, "test_at_thr": counts(yt, pt, t)}, pv, pt, yv, yt
print("helpers ready")

In [ ]:
# ===== CELL 4: one training run (skips runs that already finished) =====
def run_one(tag, backbone, strategy, seed, root=None, note=""):
    root = root or DATA
    d = f"{OUT}/runs/{tag}"
    if os.path.exists(f"{d}/result.json"):
        print("skip (done):", tag); return json.load(open(f"{d}/result.json"))
    os.makedirs(d, exist_ok=True)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)          # python, numpy and TF random state
    tr, va, te = gens(root, seed)
    y = tr.classes; cw = {0: len(y) / (2 * (y == 0).sum()), 1: len(y) / (2 * (y == 1).sum())}
    model, frac, lr = build(backbone, strategy)
    ck = f"{LOCAL}/{tag}.keras"
    t0 = time.time()
    h = model.fit(tr, epochs=EPOCHS, validation_data=va, class_weight=cw, verbose=2, callbacks=[
        ModelCheckpoint(ck, monitor='val_auroc', mode='max', save_best_only=True, verbose=0),
        EarlyStopping(monitor='val_auroc', mode='max', patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=5, min_lr=1e-8, verbose=0)])
    model = tf.keras.models.load_model(ck)        # evaluate exactly the saved best checkpoint
    res, pv, pt, yv, yt = evaluate(model, va, te)
    hist = {k: [float(v) for v in vs] for k, vs in h.history.items()}
    best_ep = int(np.argmax(hist["val_auroc"])) + 1
    res.update({"tag": tag, "backbone": backbone, "strategy": strategy, "seed": seed, "data": root, "note": note,
                "freeze": frac, "lr": lr, "epochs_run": len(hist["loss"]), "best_epoch": best_ep,
                "val_loss_at_best": hist["val_loss"][best_ep - 1], "class_weights": cw,
                "minutes": round((time.time() - t0) / 60, 1), "env": ENV})
    json.dump(hist, open(f"{d}/history.json", "w"))
    np.savez_compressed(f"{d}/scores.npz", val=pv, test=pt, y_val=yv, y_test=yt,
                        val_files=np.array(va.filenames), test_files=np.array(te.filenames))
    json.dump(res, open(f"{d}/result.json", "w"), indent=1)   # written last = run complete
    os.remove(ck)
    a = res["test_at_0.5"]; print(f"DONE {tag}: test AUROC {res['test_auroc']:.4f}  FP {a['fp']} FN {a['fn']}  ({res['minutes']} min)")
    del model; tf.keras.backend.clear_session()
    return res
print("run_one ready")

In [ ]:
# ===== FOLLOW-UP experiments (second review). Requires cells 1-4 of MultiSeed_BUSI.ipynb to have run =====
# 1. native preprocessing (keras preprocess_input instead of [0,1]) for VGG16 and ResNet50, PF and FT, seeds 0-4
# 2. mask-derived silhouette control (silhouette built from the expert mask, not from non-zero pixels), seeds 0-2
# 3. whole-image control (B-mode image without mask or crop, padded to square), seeds 0-2
# 4. determinism check: VGG16-PF seed 0 trained twice with tf.config.experimental.enable_op_determinism()
# Results go to the same Drive folder (multiseed_v1/runs) with tag prefixes native_, masksil_, whole_, det_.
# Finished runs are skipped, so this cell can be re-run after a disconnect or crash.
import gc, zipfile
from pathlib import Path
from tensorflow.keras.applications import vgg16, resnet50

NATIVE = {"VGG16": vgg16.preprocess_input, "ResNet50": resnet50.preprocess_input}
_gens_01 = gens                                   # [0,1] generators from cell 3

def gens_native(fn):
    def g(root, seed, scale=None):
        aug = ImageDataGenerator(preprocessing_function=fn, horizontal_flip=True, vertical_flip=True, zoom_range=0.20,
                                 rotation_range=15, width_shift_range=0.10, height_shift_range=0.10, fill_mode='nearest')
        plain = ImageDataGenerator(preprocessing_function=fn)
        kw = dict(target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, class_mode='binary', classes=['benign', 'malignant'])
        return (aug.flow_from_directory(f"{root}/train", shuffle=True, seed=seed, **kw),
                plain.flow_from_directory(f"{root}/val", shuffle=False, **kw),
                plain.flow_from_directory(f"{root}/test", shuffle=False, **kw))
    return g

def run(tag, backbone, strategy, seed, root=None, note="", native=False):
    global gens
    gens = gens_native(NATIVE[backbone]) if native else _gens_01
    try:
        return run_one(tag, backbone, strategy, seed, root=root, note=note)
    finally:
        gens = _gens_01
        gc.collect()

# ---- raw BUSI (same Kaggle source and primary masks as the original preprocessing) ----
RAW = "/content/BUSI_raw"
def raw_root():
    for p in Path(RAW).rglob("benign"):
        if p.is_dir() and any(p.glob("*_mask.png")):
            return p.parent
    return None
if raw_root() is None and os.path.exists("/content/drive/MyDrive/kaggle.json"):
    os.makedirs(RAW, exist_ok=True); os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.copy("/content/drive/MyDrive/kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    os.system(f"kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset -p {RAW} --force")
    with zipfile.ZipFile(f"{RAW}/breast-ultrasound-images-dataset.zip") as z:
        z.extractall(RAW)
RR = raw_root()
print("raw BUSI:", RR)

def square_resize(a):
    h, w = a.shape[:2]; m = max(h, w)
    sq = np.zeros((m, m) + a.shape[2:], dtype=np.uint8)
    sq[(m - h) // 2:(m - h) // 2 + h, (m - w) // 2:(m - w) // 2 + w] = a
    return cv2.resize(sq, (IMG_SIZE, IMG_SIZE))

def mask_silhouette(mask_path, shape):
    """Same crop box, padding and resize as the paper's preprocessing, applied to the binary mask."""
    mask = cv2.resize(cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE), (shape[1], shape[0]))
    _, b = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    x, y, w, h = cv2.boundingRect(cv2.findNonZero(b))
    px, py = int(w * 0.05), int(h * 0.05)
    crop = b[max(0, y - py):min(shape[0], y + h + py), max(0, x - px):min(shape[1], x + w + px)]
    s = square_resize(crop)
    s = ((s > 127) * 255).astype(np.uint8)
    return cv2.merge([s, s, s])

MASKSIL, WHOLE = f"{LOCAL}/BUSI_masksil", f"{LOCAL}/BUSI_whole"
if RR is not None and not os.path.exists(f"{WHOLE}/test/malignant"):
    import csv as _csv
    for r in _csv.DictReader(open(f"{OUT}/split.csv")):
        s, c, f = r["split"], r["class"], r["filename"]
        img = cv2.imread(str(RR / c / f))
        for d in (MASKSIL, WHOLE):
            os.makedirs(f"{d}/{s}/{c}", exist_ok=True)
        cv2.imwrite(f"{MASKSIL}/{s}/{c}/{f}", mask_silhouette(RR / c / f.replace(".png", "_mask.png"), img.shape))
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)          # same channel handling as the original preprocessing
        cv2.imwrite(f"{WHOLE}/{s}/{c}/{f}", square_resize(rgb))
if RR is not None:
    for d in (MASKSIL, WHOLE):
        print(d, {s: sum(len(os.listdir(f"{d}/{s}/{c}")) for c in ("benign", "malignant")) for s in ("train", "val", "test")})

# ---- runs (most informative first) ----
for sd in range(5):
    run(f"native_VGG16_PF_seed{sd}", "VGG16", "PF", sd, note="native preprocess_input", native=True)
if RR is not None:
    for b in ("DenseNet121", "VGG16"):
        for sd in (0, 1, 2):
            run(f"masksil_{b}_PF_seed{sd}", b, "PF", sd, root=MASKSIL, note="mask-derived silhouette")
            run(f"whole_{b}_PF_seed{sd}", b, "PF", sd, root=WHOLE, note="whole image, no mask")
for sd in range(5):
    run(f"native_ResNet50_PF_seed{sd}", "ResNet50", "PF", sd, note="native preprocess_input", native=True)
for b in ("VGG16", "ResNet50"):
    for sd in range(5):
        run(f"native_{b}_FT_seed{sd}", b, "FT", sd, note="native preprocess_input", native=True)

# ---- determinism check last: op determinism stays on for the rest of the session ----
tf.config.experimental.enable_op_determinism()
for rep in ("a", "b"):
    run(f"det_VGG16_PF_seed0{rep}", "VGG16", "PF", 0, note="enable_op_determinism, same seed twice")
print("FOLLOWUP ALL DONE")
